# 文档搜索中的融合检索

## 概述

此代码实现了融合检索系统，该系统将基于向量的相似性搜索与基于关键字的 BM25 检索相结合。该方法旨在利用两种方法的优势来提高文档检索的整体质量和相关性。

## 动机

传统的检索方法通常依赖于语义理解（基于向量）或关键字匹配（BM25）。每种方法都有其优点和缺点。融合检索旨在将这些方法结合起来，创建一个更强大、更准确的检索系统，可以有效地处理更广泛的查询。

## 关键组件

1. PDF 处理和文本分块
2. 使用 FAISS 和 OpenAI 嵌入支持存储创建
3. 基于关键字检索的BM25索引创建
4. 结合两种方法的自定义融合检索功能

## 方法详细信息

### 文档预处理

1. 使用 RecursiveCharacterTextSplitter 加载 PDF 并将其分割成块。
2. 通过用空格替换“t”来清理块（可能解决特定的格式问题）。

### 支持存储创建

1. OpenAI 嵌入用于创建文本块的向量表示。
2. 根据这些嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### BM25 指数创建

1. BM25 索引是根据用于支持存储的相同文本块创建的。
2. 这允许基于关键字的检索以及基于向量的方法。

### 融合检索功能

`fusion_retrieval`函数是这个实现的核心：

1. 它接受查询并执行基于向量和基于 BM25 的检索。
2. 两种方法的分数均标准化为通用量表。
3. 计算这些分数的加权组合（由 `alpha` 参数控制）。
4. 根据综合得分对文档进行排序，并返回 top-k 结果。

## 这种方法的好处

1. 提高检索质量：通过结合语义搜索和基于关键字的搜索，系统可以捕获概念相似性和精确的关键字匹配。
2. 灵活性：`alpha` 参数允许根据特定用例或查询类型调整向量搜索和关键字搜索之间的平衡。
3. 鲁棒性：组合方法可以有效地处理更广泛的查询，从而减轻单个方法的弱点。
4. 可定制性：系统可以轻松适应使用不同的支持存储或基于关键字的检索方法。

## 结论

融合检索代表了一种强大的文档搜索方法，它结合了语义理解和关键字匹配的优势。通过利用基于向量和BM25检索方法，它为信息检索任务提供了更全面、更灵活的解决方案。这种方法在概念相似性和关键词相关性都很重要的各个领域都有潜在的应用，例如学术研究、法律文档搜索或通用搜索引擎。

<div style="text-align: center;">

<img src="../images/fusion_retrieval.svg" alt="Fusion Retrieval" style="width:100%; height:auto;">
</div>

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install langchain numpy python-dotenv rank-bm25

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain.docstore.document import Document

from typing import List
from rank_bm25 import BM25Okapi
import numpy as np


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### 定义文档路径

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [2]:
path = "data/Understanding_Climate_Change.pdf"

### 将 pdf 编码为支持存储并返回之前步骤中的拆分文档以创建 BM25 实例

In [3]:
def encode_pdf_and_get_split_documents(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore, cleaned_texts

### 创建向量存储并获取分块文档

In [4]:
vectorstore, cleaned_texts = encode_pdf_and_get_split_documents(path)

### 创建bm25索引用于通过关键字检索文档

In [5]:
def create_bm25_index(documents: List[Document]) -> BM25Okapi:
    """
    Create a BM25 index from the given documents.

    BM25 (Best Matching 25) is a ranking function used in information retrieval.
    It's based on the probabilistic retrieval framework and is an improvement over TF-IDF.

    Args:
    documents (List[Document]): List of documents to index.

    Returns:
    BM25Okapi: An index that can be used for BM25 scoring.
    """
    # Tokenize each document by splitting on whitespace
    # This is a simple approach and could be improved with more sophisticated tokenization
    tokenized_docs = [doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_docs)

In [6]:
bm25 = create_bm25_index(cleaned_texts) # Create BM25 index from the cleaned texts (chunks)

### 定义一个函数，可以按语义和关键字检索，标准化分数并获取前 k 个文档

In [7]:
def fusion_retrieval(vectorstore, bm25, query: str, k: int = 5, alpha: float = 0.5) -> List[Document]:
    """
    Perform fusion retrieval combining keyword-based (BM25) and vector-based search.

    Args:
    vectorstore (VectorStore): The vectorstore containing the documents.
    bm25 (BM25Okapi): Pre-computed BM25 index.
    query (str): The query string.
    k (int): The number of documents to retrieve.
    alpha (float): The weight for vector search scores (1-alpha will be the weight for BM25 scores).

    Returns:
    List[Document]: The top k documents based on the combined scores.
    """
    
    epsilon = 1e-8

    # Step 1: Get all documents from the vectorstore
    all_docs = vectorstore.similarity_search("", k=vectorstore.index.ntotal)

    # Step 2: Perform BM25 search
    bm25_scores = bm25.get_scores(query.split())

    # Step 3: Perform vector search
    vector_results = vectorstore.similarity_search_with_score(query, k=len(all_docs))
    
    # Step 4: Normalize scores
    vector_scores = np.array([score for _, score in vector_results])
    vector_scores = 1 - (vector_scores - np.min(vector_scores)) / (np.max(vector_scores) - np.min(vector_scores) + epsilon)

    bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) -  np.min(bm25_scores) + epsilon)

    # Step 5: Combine scores
    combined_scores = alpha * vector_scores + (1 - alpha) * bm25_scores  

    # Step 6: Rank documents
    sorted_indices = np.argsort(combined_scores)[::-1]
    
    # Step 7: Return top k documents
    return [all_docs[i] for i in sorted_indices[:k]]

### 用例示例

In [8]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
top_docs = fusion_retrieval(vectorstore, bm25, query, k=5, alpha=0.5)
docs_content = [doc.page_content for doc in top_docs]
show_context(docs_content)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--fusion-retrieval)